## 1. Accessing the data in the JSON

In [1]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [12]:
# Get the weather for 5 days for Berlin.
# First define the latitude, longitude and API_key.
latitude = 52.5200
longitude = 13.405
API_key = os.getenv("API_key")
if API_key is None:
    print("Error: API_key not found in .env file!")
else:
    print(f"Key loaded successfully: {API_key[:5]}...") # prints first 5 characters

# Combine the different parts from above to create one url.
weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={latitude}&lon={longitude}&appid={API_key}", timeout=5)
weather_json = weather.json()
weather_json

Key loaded successfully: 50d72...


{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1772550000,
   'main': {'temp': 289.33,
    'feels_like': 288.34,
    'temp_min': 289.15,
    'temp_max': 289.33,
    'pressure': 1026,
    'sea_level': 1026,
    'grnd_level': 1020,
    'humidity': 51,
    'temp_kf': 0.18},
   'weather': [{'id': 800,
     'main': 'Clear',
     'description': 'clear sky',
     'icon': '01d'}],
   'clouds': {'all': 5},
   'wind': {'speed': 2.18, 'deg': 249, 'gust': 3.63},
   'visibility': 10000,
   'pop': 0,
   'sys': {'pod': 'd'},
   'dt_txt': '2026-03-03 15:00:00'},
  {'dt': 1772560800,
   'main': {'temp': 283.66,
    'feels_like': 282.33,
    'temp_min': 280.78,
    'temp_max': 283.66,
    'pressure': 1026,
    'sea_level': 1026,
    'grnd_level': 1021,
    'humidity': 60,
    'temp_kf': 2.88},
   'weather': [{'id': 801,
     'main': 'Clouds',
     'description': 'few clouds',
     'icon': '02n'}],
   'clouds': {'all': 11},
   'wind': {'speed': 1.75, 'deg': 292, 'gust': 1.72},
   'visibility'

Let's access a few values from the JSON.

In [13]:
# Which are the keys?
weather_json.keys()

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

In [14]:
# What is the value of the list key?
weather_json["list"]

[{'dt': 1772550000,
  'main': {'temp': 289.33,
   'feels_like': 288.34,
   'temp_min': 289.15,
   'temp_max': 289.33,
   'pressure': 1026,
   'sea_level': 1026,
   'grnd_level': 1020,
   'humidity': 51,
   'temp_kf': 0.18},
  'weather': [{'id': 800,
    'main': 'Clear',
    'description': 'clear sky',
    'icon': '01d'}],
  'clouds': {'all': 5},
  'wind': {'speed': 2.18, 'deg': 249, 'gust': 3.63},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2026-03-03 15:00:00'},
 {'dt': 1772560800,
  'main': {'temp': 283.66,
   'feels_like': 282.33,
   'temp_min': 280.78,
   'temp_max': 283.66,
   'pressure': 1026,
   'sea_level': 1026,
   'grnd_level': 1021,
   'humidity': 60,
   'temp_kf': 2.88},
  'weather': [{'id': 801,
    'main': 'Clouds',
    'description': 'few clouds',
    'icon': '02n'}],
  'clouds': {'all': 11},
  'wind': {'speed': 1.75, 'deg': 292, 'gust': 1.72},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'n'},
  'dt_txt': '2026-03-03 18:00:00'},
 {'dt':

In [15]:
# What is the wind?
# [0] refers to the first 3-hour block in the forecast
wind_speed = weather_json["list"][0]["wind"]["speed"]
print(f"Wind Speed: {wind_speed} m/s")

Wind Speed: 2.18 m/s


## 2.  Transforming a JSON into a DataFrame

🚲 What’s useful for E-Scooter Users?
* *dt_txt*: data about date
* *main.temp*: data about temperature
* *main.feels_like*: how the temperature feels like. The "RealFeel" tells them if they need gloves to avoid numb fingers
* *wind.speed*: data about wind's speed
* *wind.gust*: more critical than speed. A 14.47 m/s gust (found in Feb 21 data) can easily knock a standing rider off balance. 
* *visibility*: your data shows a drop to 5299m during rain. Lower visibility means a higher risk of not being seen by cars
* *weather.main*: clouds, rain etc.
* *weather.description*: clouds, rain etc. and description
* *pop (Probability of Precipitation)*: this is the #1 "Abandon Trip" metric. If pop > 0.5, users will likely switch to public transport
* *rain*: crucial for safety. 
* *snow*: crucial for safety. Even 0.31mm of snow makes city tiles and metal manhole covers lethal for thin scooter tires

In [16]:
# Flatten the JSON
weather_df = pd.json_normalize(weather_json['list'])

# Extract nested list data (the 'weather' column is a list)
weather_df['weather.main'] = weather_df['weather'].str[0].str.get('main')
weather_df['weather.description'] = weather_df['weather'].str[0].str.get('description')

# List every useful column in final table
target_cols = [
    'dt_txt', 'main.temp', 'main.feels_like', 'wind.speed',
    'wind.gust', 'visibility', 'weather.main',
    'weather.description', 'pop', 'rain', 'snow'
]

# Create clean_df. fill_value=0 handles missing rain/snow keys automatically.
clean_df = weather_df.reindex(columns=target_cols, fill_value=0)

#Final cleanup: replace any NaNs within existing columns with 0
clean_df = clean_df.fillna(0)

clean_df

,dt_txt,main.temp,main.feels_like,wind.speed,wind.gust,visibility,weather.main,weather.description,pop,rain,snow
0,2026-03-03 15:00:00,289.33,288.34,2.18,3.63,10000,Clear,clear sky,0,0,0
1,2026-03-03 18:00:00,283.66,282.33,1.75,1.72,10000,Clouds,few clouds,0,0,0
2,2026-03-03 21:00:00,277.54,276.09,1.75,1.89,10000,Clouds,few clouds,0,0,0
3,2026-03-04 00:00:00,275.61,273.22,2.32,5.62,10000,Clouds,few clouds,0,0,0
4,2026-03-04 03:00:00,278.87,277.23,2.13,4.62,10000,Clouds,few clouds,0,0,0
5,2026-03-04 06:00:00,277.67,276.16,1.82,3.83,10000,Clouds,few clouds,0,0,0
6,2026-03-04 09:00:00,280.12,278.61,2.22,3.17,10000,Clouds,few clouds,0,0,0
7,2026-03-04 12:00:00,283.24,281.22,2.33,3.09,10000,Clear,clear sky,0,0,0
8,2026-03-04 15:00:00,283.91,281.93,2.23,2.71,10000,Clear,clear sky,0,0,0
9,2026-03-04 18:00:00,280.99,280.44,1.41,1.44,10000,Clear,clear sky,0,0,0


In [17]:
# Transform JSON into a DataFrame. Option #2
weather_df = pd.DataFrame(weather_json['list'])
weather_df

,dt,main,weather,clouds,wind,visibility,pop,sys,dt_txt
0,1772550000,"{'temp': 289.33, 'feels_like': 288.34, 'temp_m...","[{'id': 800, 'main': 'Clear', 'description': '...",{'all': 5},"{'speed': 2.18, 'deg': 249, 'gust': 3.63}",10000,0,{'pod': 'd'},2026-03-03 15:00:00
1,1772560800,"{'temp': 283.66, 'feels_like': 282.33, 'temp_m...","[{'id': 801, 'main': 'Clouds', 'description': ...",{'all': 11},"{'speed': 1.75, 'deg': 292, 'gust': 1.72}",10000,0,{'pod': 'n'},2026-03-03 18:00:00
2,1772571600,"{'temp': 277.54, 'feels_like': 276.09, 'temp_m...","[{'id': 801, 'main': 'Clouds', 'description': ...",{'all': 15},"{'speed': 1.75, 'deg': 323, 'gust': 1.89}",10000,0,{'pod': 'n'},2026-03-03 21:00:00
3,1772582400,"{'temp': 275.61, 'feels_like': 273.22, 'temp_m...","[{'id': 801, 'main': 'Clouds', 'description': ...",{'all': 16},"{'speed': 2.32, 'deg': 5, 'gust': 5.62}",10000,0,{'pod': 'n'},2026-03-04 00:00:00
4,1772593200,"{'temp': 278.87, 'feels_like': 277.23, 'temp_m...","[{'id': 801, 'main': 'Clouds', 'description': ...",{'all': 15},"{'speed': 2.13, 'deg': 4, 'gust': 4.62}",10000,0,{'pod': 'n'},2026-03-04 03:00:00
5,1772604000,"{'temp': 277.67, 'feels_like': 276.16, 'temp_m...","[{'id': 801, 'main': 'Clouds', 'description': ...",{'all': 12},"{'speed': 1.82, 'deg': 4, 'gust': 3.83}",10000,0,{'pod': 'd'},2026-03-04 06:00:00
6,1772614800,"{'temp': 280.12, 'feels_like': 278.61, 'temp_m...","[{'id': 801, 'main': 'Clouds', 'description': ...",{'all': 18},"{'speed': 2.22, 'deg': 350, 'gust': 3.17}",10000,0,{'pod': 'd'},2026-03-04 09:00:00
7,1772625600,"{'temp': 283.24, 'feels_like': 281.22, 'temp_m...","[{'id': 800, 'main': 'Clear', 'description': '...",{'all': 9},"{'speed': 2.33, 'deg': 342, 'gust': 3.09}",10000,0,{'pod': 'd'},2026-03-04 12:00:00
8,1772636400,"{'temp': 283.91, 'feels_like': 281.93, 'temp_m...","[{'id': 800, 'main': 'Clear', 'description': '...",{'all': 0},"{'speed': 2.23, 'deg': 342, 'gust': 2.71}",10000,0,{'pod': 'd'},2026-03-04 15:00:00
9,1772647200,"{'temp': 280.99, 'feels_like': 280.44, 'temp_m...","[{'id': 800, 'main': 'Clear', 'description': '...",{'all': 3},"{'speed': 1.41, 'deg': 341, 'gust': 1.44}",10000,0,{'pod': 'n'},2026-03-04 18:00:00


## 3. Create a function for 3 cities

In [18]:
cities = ['Berlin', 'Hamburg', 'Munich']
country = 'Germany'
API_key = os.getenv("API_key")
weather_data = []
# Combine the different parts from above to create one url

for city in cities:
    # Geocoding: get lat/lon for any city/country
    geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country}&limit=1&appid={API_key}"
    geo_response = requests.get(geo_url, timeout=5)

    if geo_response.status_code == 200 and geo_response.json():
        geo_data = geo_response.json()[0]
        lat, lon = geo_data['lat'], geo_data['lon']

        # Get weather using a specific lat/lon
        weather_url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
        weather_response = requests.get(weather_url, timeout=5)

        if weather_response.status_code == 200:
            weather_json = weather_response.json()

        # Transform JSON into DataFrame
        temp_df = pd.json_normalize(weather_json['list'])
        temp_df['city'] = city

        # Extract nested list data (the 'weather' column is a list)
        temp_df['weather.main'] = temp_df['weather'].str[0].str.get('main')
        temp_df['weather.description'] = temp_df['weather'].str[0].str.get('description')

        weather_data.append(temp_df)

# Combine all cities into one DataFrame
if weather_data:
     weather_df = pd.concat(weather_data, ignore_index=True)
    # Handle Rain and Snow (OpenWeather uses 'rain.3h' and 'snow.3h')
target_cols = ['city',
    'dt_txt', 'main.temp', 'main.feels_like', 'wind.speed',
    'wind.gust', 'visibility', 'weather.main',
    'weather.description', 'pop', 'rain', 'snow'
]
    # Create clean_df. fill_value=0 handles missing rain/snow keys automatically
clean_df = weather_df.reindex(columns=target_cols, fill_value=0).fillna(0)
clean_df
print(clean_df.to_string())


        city               dt_txt  main.temp  main.feels_like  wind.speed  wind.gust  visibility weather.main weather.description  pop  rain  snow
0     Berlin  2026-03-03 15:00:00      16.10            15.07        2.18       3.63       10000        Clear           clear sky    0     0     0
1     Berlin  2026-03-03 18:00:00      10.27             8.97        1.75       1.72       10000       Clouds          few clouds    0     0     0
2     Berlin  2026-03-03 21:00:00       4.39             2.94        1.75       1.89       10000       Clouds          few clouds    0     0     0
3     Berlin  2026-03-04 00:00:00       2.46             0.07        2.32       5.62       10000       Clouds          few clouds    0     0     0
4     Berlin  2026-03-04 03:00:00       5.72             4.08        2.13       4.62       10000       Clouds          few clouds    0     0     0
5     Berlin  2026-03-04 06:00:00       4.52             3.01        1.82       3.83       10000       Clouds         

Create a function to get weather for defined cities.

In [19]:
def get_weather_forecast(cities, country, api_key):
    weather_data = []

    for city in cities:
        # geocoding
        geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country}&limit=1&appid={api_key}"
        geo_response = requests.get(geo_url, timeout=5)

        if geo_response.status_code == 200 and geo_response.json():
            geo_data = geo_response.json()[0]
            lat, lon = geo_data['lat'], geo_data['lon']

            # weather Forecast
            weather_url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric"
            weather_response = requests.get(weather_url, timeout=5)

            if weather_response.status_code == 200:
                weather_json = weather_response.json()

                # transform
                temp_df = pd.json_normalize(weather_json['list'])
                temp_df['city'] = city
                temp_df['weather.main'] = temp_df['weather'].str[0].str.get('main')
                temp_df['weather.description'] = temp_df['weather'].str[0].str.get('description')

                weather_data.append(temp_df)

    # combine and clean
    if not weather_data:
        return pd.DataFrame() # Return empty DF if no data found

    weather_df = pd.concat(weather_data, ignore_index=True)

    target_cols = [
        'city', 'dt_txt', 'main.temp', 'main.feels_like', 'wind.speed',
        'wind.gust', 'visibility', 'weather.main',
        'weather.description', 'pop', 'rain', 'snow'
    ]

    return weather_df.reindex(columns=target_cols, fill_value=0).fillna(0)

# --- How to use the function ---
my_cities = ['Berlin', 'Hamburg', 'Munich']
my_key = os.getenv("API_key")

final_weather_df = get_weather_forecast(my_cities, 'Germany', my_key)
final_weather_df


,city,dt_txt,main.temp,main.feels_like,wind.speed,wind.gust,visibility,weather.main,weather.description,pop,rain,snow
0,Berlin,2026-03-03 15:00:00,16.10,15.07,2.18,3.63,10000,Clear,clear sky,0,0,0
1,Berlin,2026-03-03 18:00:00,10.27,8.97,1.75,1.72,10000,Clouds,few clouds,0,0,0
2,Berlin,2026-03-03 21:00:00,4.39,2.94,1.75,1.89,10000,Clouds,few clouds,0,0,0
3,Berlin,2026-03-04 00:00:00,2.46,0.07,2.32,5.62,10000,Clouds,few clouds,0,0,0
4,Berlin,2026-03-04 03:00:00,5.72,4.08,2.13,4.62,10000,Clouds,few clouds,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
115,Munich,2026-03-08 00:00:00,6.16,6.16,0.42,0.62,10000,Clear,clear sky,0,0,0
116,Munich,2026-03-08 03:00:00,5.21,5.21,0.80,0.72,10000,Clear,clear sky,0,0,0
117,Munich,2026-03-08 06:00:00,4.63,4.63,1.11,1.09,10000,Clear,clear sky,0,0,0
118,Munich,2026-03-08 09:00:00,10.21,9.09,0.18,0.19,10000,Clouds,scattered clouds,0,0,0


Rename the columns in our DataFrame:

In [20]:
# Define a dictionary mapping the old names to new names
column_mapping = {
    'city': 'city',
    'dt_txt': 'forecast_time',
    'main.temp': 'temperature',
    'main.feels_like': 'feels_like',
    'wind.speed': 'wind_speed',
    'wind.gust': 'wind_gust',
    'visibility': 'visibility',
    'weather.main': 'weather',
    'weather.description': 'description_',
    'pop': 'precipitation_prob',
    'rain.3h': 'rain',
    'snow.3h': 'snow'
       }

# Apply the rename
weathers_df = clean_df.rename(columns=column_mapping)

# View the final result
weathers_df

,city,forecast_time,temperature,feels_like,wind_speed,wind_gust,visibility,weather,description_,precipitation_prob,rain,snow
0,Berlin,2026-03-03 15:00:00,16.10,15.07,2.18,3.63,10000,Clear,clear sky,0,0,0
1,Berlin,2026-03-03 18:00:00,10.27,8.97,1.75,1.72,10000,Clouds,few clouds,0,0,0
2,Berlin,2026-03-03 21:00:00,4.39,2.94,1.75,1.89,10000,Clouds,few clouds,0,0,0
3,Berlin,2026-03-04 00:00:00,2.46,0.07,2.32,5.62,10000,Clouds,few clouds,0,0,0
4,Berlin,2026-03-04 03:00:00,5.72,4.08,2.13,4.62,10000,Clouds,few clouds,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
115,Munich,2026-03-08 00:00:00,6.16,6.16,0.42,0.62,10000,Clear,clear sky,0,0,0
116,Munich,2026-03-08 03:00:00,5.21,5.21,0.80,0.72,10000,Clear,clear sky,0,0,0
117,Munich,2026-03-08 06:00:00,4.63,4.63,1.11,1.09,10000,Clear,clear sky,0,0,0
118,Munich,2026-03-08 09:00:00,10.21,9.09,0.18,0.19,10000,Clouds,scattered clouds,0,0,0


## 4. Retrieving our DataFrame to SQL

In [21]:
# configuration and connection
schema = "sql_gans"
host = "127.0.0.1"
user = "root"
password = os.getenv("password")
port = 3306

if password is None:
    raise ValueError("ERROR: 'password' not found in .env file. Check your filename and path.")

# build connection string only if password exists
connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
print("Connection string created successfully.")

Connection string created successfully.


Merging our weather DataFrame with `cities` table.

In [22]:
from sqlalchemy import create_engine
# fetch the city_ids from MySQL 'cities' table
engine = create_engine(connection_string)
cities_from_sql = pd.read_sql_table('cities', con=engine)
final_to_sql = (
    weathers_df
    .merge(cities_from_sql[['city_id', 'city']], on='city', how='left')
)

Create the function to send data from our DataFrame to `weathers` table into MySQL

In [23]:
def send_weather_to_sql(weathers_df, engine):
        # create engine inside the function

        # update the list: 'city_id' instead of 'city'
        sql_columns = [
                'city_id', 'forecast_time', 'temperature', 'feels_like',
                'wind_speed', 'wind_gust', 'visibility', 'weather', 'description_',
                'precipitation_prob', 'rain', 'snow'
        ]

        try:
          data_to_send = weathers_df[sql_columns]
          data_to_send.to_sql(
            name='weathers',      # specify the SQL table name here
            con=engine,           # database connection
            if_exists='append',   # add data to existing table
            index=False           # do not write the DataFrame index as a column
            )
          print(f"Success! {len(data_to_send)} rows synced to SQL.")

        except KeyError as e:
          print(f"Error: Your DataFrame is missing a column: {e}")
        except Exception as e:
          print(f"SQL Error: {e}")
send_weather_to_sql(final_to_sql, engine)

Success! 120 rows synced to SQL.


Create a function which fetching data from `cities` table in MySQL, call weather API and send data to MySQL (into `weathers` table)

In [24]:
# configuration and connection
schema = "sql_gans"
host = "127.0.0.1"
user = "root"
password = os.getenv("password")
port = 3306

if password is None:
    raise ValueError("ERROR: 'password' not found in .env file. Check your filename and path.")

# build connection string only if password exists
connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
print("Connection string created successfully.")

Connection string created successfully.


In [25]:
import pandas as pd
import requests
import os
from sqlalchemy import create_engine

def get_weather_forecast(connection_string, API_key):
    engine = create_engine(connection_string)

    # fetch City Data
    cities_df = pd.read_sql("SELECT city_id, city, country FROM cities", con=engine)
    print(f"--- Found {len(cities_df)} cities in SQL table ---")

    if cities_df.empty:
        print("Error: Your 'cities' table is empty!")
        return

    weather_data = []

    for _, row in cities_df.iterrows():
        city = row['city']
        country = row['country']
        city_id = row['city_id']
        print(f"Processing: {city}, {country}...")

        # geocoding URL path
        geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country}&limit=1&appid={API_key}"
        geo_response = requests.get(geo_url)

        if geo_response.status_code != 200:
            print(f"API Error {geo_response.status_code}: {geo_response.text}")
            continue
        geo_data = geo_response.json()
        # safe access to list elements
        if geo_data and isinstance(geo_data, list) and len(geo_data) > 0:
            lat = geo_data[0]['lat']
            lon = geo_data[0]['lon']

            # weather URL path
            weather_url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
            weather_response = requests.get(weather_url).json()

            if 'list' in weather_response:
                temp_df = pd.json_normalize(weather_response['list'])
                temp_df['city_id'] = city_id

                # unpack nested weather description safely
                temp_df['weather.main'] = temp_df['weather'].str[0].str.get('main')
                temp_df['weather.description'] = temp_df['weather'].str[0].str.get('description')

                weather_data.append(temp_df)
                print(f"Success for {city}")
            else:
                print(f"Forecast failed for {city}: {weather_response.get('message', 'No message')}")
        else:
            print(f"Geocoding failed: Could not find coordinates for '{city}'")

    if not weather_data:
        print("No weather data found.")
        return

    full_df = pd.concat(weather_data, ignore_index=True)

    column_mapping = {
        'dt_txt': 'forecast_time',
        'main.temp': 'temperature',
        'main.feels_like': 'feels_like',
        'wind.speed': 'wind_speed',
        'wind.gust': 'wind_gust',
        'visibility': 'visibility',
        'weather.main': 'weather',
        'weather.description': 'description_',
        'pop': 'precipitation_prob',
        'rain.3h': 'rain',
        'snow.3h': 'snow'
    }

    # prevent KeyError if rain/snow columns are missing from the API response
    final_df = (
        full_df.reindex(columns=list(column_mapping.keys()) + ['city_id'], fill_value=0)
        .rename(columns=column_mapping)
        .fillna(0)
    )

    final_df.to_sql(name='weathers', con=engine, if_exists='append', index=False)
    print(f"Successfully updated weather for {len(weather_data)} cities.")

# --- Execution ---
API_key = os.getenv("API_key")
# (Your connection_string logic here...)
# configuration and connection
schema = "sql_gans"
host = "127.0.0.1"
user = "root"
password = os.getenv("password")
port = 3306

if password is None:
    raise ValueError("ERROR: 'password' not found in .env file. Check your filename and path.")

# build connection string only if password exists
connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
print("Connection string created successfully.")

get_weather_forecast(connection_string, API_key)


Connection string created successfully.
--- Found 3 cities in SQL table ---
Processing: Berlin, Germany...
Success for Berlin
Processing: Hamburg, Germany...
Success for Hamburg
Processing: Munich, Germany...
Success for Munich
Successfully updated weather for 3 cities.
